# 01 — Exploratory Data Analysis & Feature EngineeringThis notebook loads the two peptide absorbance trials, averages them,extracts sequence-derived features, and explores distributions and correlations.**Data**: 96 peptide sequences, each with absorbance readings at the last 20 time points across two experimental trials.

In [ ]:
import sys, ossys.path.insert(0, os.path.abspath('..'))import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom src.data_loading import load_trial_data, average_trials, get_absorbance_columnsfrom src.feature_extraction import extract_all_features, extract_features_dataframefrom src.visualization import (    plot_feature_distributions,    plot_correlation_heatmap,)%matplotlib inlineplt.rcParams['figure.dpi'] = 150

## 1. Load and Average Trials

In [ ]:
DATA_DIR = os.path.join('..', 'data')df1, df2 = load_trial_data(    os.path.join(DATA_DIR, 'peptide_csv1_last20.csv'),    os.path.join(DATA_DIR, 'peptide2_csv1_last20.csv'),)print(f'Trial 1 shape: {df1.shape}')print(f'Trial 2 shape: {df2.shape}')print(f'Columns match: {list(df1.columns) == list(df2.columns)}')print(f'Sequences match: {(df1["Sequence"] == df2["Sequence"]).all()}')df1.head()

In [ ]:
df_avg = average_trials(df1, df2)df_avg[['Sequence', 'mean_abs', 'std_abs', 'cv', 'max_abs', 'min_abs']].head(10)

## 2. Extract Sequence FeaturesWe extract biophysical, compositional, positional, and physicochemicalfeatures from each peptide sequence using BioPython's ProteinAnalysis.

In [ ]:
sequences = df_avg['Sequence'].tolist()df_features = extract_features_dataframe(sequences)print(f'Extracted {df_features.shape[1]} features for {df_features.shape[0]} sequences')df_features.head()

In [ ]:
# Merge features with targetdf_model = pd.concat([    df_avg[['Sequence', 'mean_abs', 'std_abs', 'cv']].reset_index(drop=True),    df_features.reset_index(drop=True),], axis=1)print(f'Combined DataFrame shape: {df_model.shape}')df_model.head()

## 3. Feature Distributions

In [ ]:
core_features = [    'seq_length', 'gravy', 'isoelectric_point', 'aromaticity',    'instability_index', 'mol_weight', 'helix_frac', 'sheet_frac', 'turn_frac',]fig = plot_feature_distributions(df_model, core_features)plt.show()

In [ ]:
plt.figure(figsize=(8, 4), dpi=150)sns.boxplot(data=df_model[[    'hydrophobic_ratio', 'charged_ratio', 'polar_ratio', 'aromatic_ratio',    'positive_charge_ratio', 'negative_charge_ratio',]])plt.xticks(rotation=45, fontsize=8)plt.title('Charge & Hydrophobicity Ratios')plt.tight_layout()plt.show()

## 4. Correlation Analysis

In [ ]:
fig = plot_correlation_heatmap(df_model, core_features, target='mean_abs')plt.show()

In [ ]:
# Top correlations with mean absorbancecorr = df_model[df_features.columns.tolist() + ['mean_abs']].corr()['mean_abs']corr = corr.drop('mean_abs').sort_values(ascending=False)print('Top 10 positive correlations:')print(corr.head(10))print('\nTop 10 negative correlations:')print(corr.tail(10))

In [ ]:
# Pairplot of key featuressample = df_model[['gravy', 'mol_weight', 'instability_index', 'isoelectric_point', 'mean_abs']]sns.pairplot(sample)plt.suptitle('Pairwise Feature Relationships', y=1.02)plt.show()

## 5. Save Processed DataSave the combined feature + target DataFrame for use in modeling notebooks.

In [ ]:
df_model.to_csv(os.path.join(DATA_DIR, 'processed_features.csv'), index=False)print(f'Saved processed data: {df_model.shape}')